# 04_任务列表 (Todo List)

**来源:** [LangChain Docs — Todo List Frontend Pattern](https://docs.langchain.com/deep-agents/frontend/todo-list)

并非每次智能体交互都是聊天。有时智能体在执行多步骤计划，展示进度的最佳方式是实时更新的任务列表。本教程演示如何直接从智能体状态读取 `todos` 数组，在智能体执行计划时实时渲染每个项目的当前状态。

## 1. 工作原理

在 LangGraph 智能体中，状态不仅限于消息。可以定义保存任意数据的自定义状态键——在本例中是 `todos` 数组。智能体执行计划时，将每个 todo 的状态从 `pending` 更新为 `in_progress` 再到 `completed`。

```
用户提交请求 → 智能体创建计划并填充 todos → 
智能体执行每个 todo → 状态流转：pending → in_progress → completed → 
stream.values.todos 实时更新 → UI 重新渲染
```

## 2. 设置 useStream

无需特殊配置，只需将 useStream 指向智能体，然后从 `stream.values` 读取 todos。

In [ ]:
# TypeScript 类型定义：
#
# import type { BaseMessage } from "@langchain/core/messages";
#
# interface TodoItem {
#   title: string;
#   status: "pending" | "in_progress" | "completed";
#   description?: string;
# }
#
# interface AgentState {
#   messages: BaseMessage[];
#   todos: TodoItem[];
# }

print("AgentState 中的 todos 数组是自定义状态键，用于实时跟踪进度")

### React 集成示例

In [ ]:
# React 前端集成：
#
# import { useStream } from "@langchain/react";
#
# const AGENT_URL = "http://localhost:2024";
#
# export function TodoAgent() {
#   const stream = useStream<typeof myAgent>({
#     apiUrl: AGENT_URL,
#     assistantId: "deep_agent_todo_list",
#   });
#
#   const todos = stream.values?.todos ?? [];
#
#   return (
#     <div>
#       <TodoList todos={todos} />
#       {stream.messages.map((msg) => (
#         <Message key={msg.id} message={msg} />
#       ))}
#     </div>
#   );
# }

print("stream.values?.todos 实时反映智能体的状态变更")

## 3. Todo 接口

| 属性 | 类型 | 说明 |
|------|------|------|
| `status` | `pending \| in_progress \| completed` | 任务的当前状态 |
| `content` | `string` | 任务内容的人类可读描述 |

## 4. 构建 TodoList 组件

任务列表渲染每个项目，包含状态图标、颜色编码和视觉样式。

In [ ]:
# React TodoList 组件：
#
# function TodoList({ todos }: { todos: Todo[] }) {
#   const completed = todos.filter((t) => t.status === "completed").length;
#   const percentage = todos.length ? Math.round((completed / todos.length) * 100) : 0;
#
#   return (
#     <div className="rounded-lg border bg-white p-4 shadow-sm">
#       <div className="mb-4 flex items-center justify-between">
#         <h2 className="text-lg font-semibold">Agent Progress</h2>
#         <span className="text-sm text-gray-500">{completed}/{todos.length} tasks</span>
#       </div>
#       <ProgressBar percentage={percentage} />
#       <ul className="mt-4 space-y-2">
#         {todos.map((todo, i) => (
#           <TodoItem key={i} todo={todo} />
#         ))}
#       </ul>
#     </div>
#   );
# }

print("TodoList：顶部显示进度百分比 + 完成计数，下方列出每个任务项")

### 进度条

In [ ]:
# React 进度条：
#
# function ProgressBar({ percentage }: { percentage: number }) {
#   return (
#     <div className="space-y-1">
#       <div className="flex items-center justify-between text-xs text-gray-500">
#         <span>Progress</span>
#         <span>{percentage}%</span>
#       </div>
#       <div className="h-2 overflow-hidden rounded-full bg-gray-200">
#         <div className="h-full rounded-full bg-green-500 transition-all duration-500"
#           style={{ width: `${percentage}%` }} />
#       </div>
#     </div>
#   );
# }

print("ProgressBar：百分比进度条，过渡动画 500ms")

### 单个 Todo 项目

每个项目有状态图标、颜色编码文本，已完成任务添加删除线样式。

In [ ]:
# React TodoItem 组件：
#
# function TodoItem({ todo }: { todo: Todo }) {
#   const config = {
#     pending:    { icon: "○", textClass: "text-gray-600",    bgClass: "bg-gray-50",               iconClass: "text-gray-400" },
#     in_progress:{ icon: "◉", textClass: "text-amber-800",  bgClass: "bg-amber-50 border-amber-200", iconClass: "text-amber-500 animate-pulse" },
#     completed:  { icon: "✓", textClass: "text-green-800 line-through", bgClass: "bg-green-50 border-green-200", iconClass: "text-green-500" },
#   };
#   const style = config[todo.status];
#
#   return (
#     <li className={`flex items-start gap-3 rounded-md border px-3 py-2 ${style.bgClass}`}>
#       <span className={`mt-0.5 text-lg leading-none ${style.iconClass}`}>{style.icon}</span>
#       <span className={`text-sm ${style.textClass}`}>{todo.content}</span>
#     </li>
#   );
# }

print("TodoItem：pending→○灰色，in_progress→◉琥珀色脉冲，completed→✓绿色删除线")

## 5. 与聊天消息结合

任务列表与常规聊天界面并存。实用布局是将任务列表作为持久侧栏或顶部面板，下方显示聊天消息。

In [ ]:
# React 组合布局：
#
# function TodoAgentLayout() {
#   const stream = useStream<typeof myAgent>({
#     apiUrl: AGENT_URL,
#     assistantId: "deep_agent_todo_list",
#   });
#   const todos = stream.values?.todos ?? [];
#
#   return (
#     <div className="flex h-screen flex-col">
#       {todos.length > 0 && (
#         <div className="border-b bg-gray-50 p-4">
#           <TodoList todos={todos} />
#         </div>
#       )}
#       <main className="flex-1 overflow-y-auto p-6">
#         <div className="mx-auto max-w-2xl space-y-4">
#           {stream.messages.map((msg) => (
#             <Message key={msg.id} message={msg} />
#           ))}
#         </div>
#       </main>
#       <ChatInput onSubmit={(text) => stream.submit({messages:[{type:"human",content:text}]})} isLoading={stream.isLoading} />
#     </div>
#   );
# }

print("todo 列表为空时隐藏，智能体创建计划后自动显示")

## 6. 自定义状态扩展

此模式展示了一个强大原则：`stream.values` 可以暴露智能体定义的任何自定义状态，而不仅仅是消息。

| 示例 | 用法 |
|------|------|
| 进度指标 | `stream.values.progress` |
| 生成的作品 | `stream.values.document` |
| 决策日志 | `stream.values.decisions` |
| 资源列表 | `stream.values.sources` |
| 置信度分数 | `stream.values.confidence_score` |

## 7. 动画过渡

Todo 状态转换实时发生，平滑动画让变更更自然：

- `transition-all duration-300 ease-in-out` — 颜色、删除线、透明度的平滑过渡
- `animate-pulse` — `in_progress` 图标脉冲效果，吸引注意力
- 仅高亮一个 `in_progress` 项目，避免 UI 杂乱

## 8. 空状态与加载状态处理

In [ ]:
# React 空状态处理：
#
# function TodoList({ todos, isLoading }) {
#   if (todos.length === 0 && !isLoading) return null;
#
#   if (todos.length === 0 && isLoading) {
#     return (
#       <div className="rounded-lg border bg-white p-4 shadow-sm">
#         <div className="flex items-center gap-2 text-sm text-gray-500">
#           <span className="animate-spin">⟳</span>
#           Agent is creating a plan...
#         </div>
#       </div>
#     );
#   }
#   return (/* ... full todo list rendering */);
# }

print("空状态：智能体还未创建计划时显示旋转加载提示")

## 9. 最佳实践

| 实践 | 说明 |
|------|------|
| 突出显示任务列表 | 作为计划型智能体的主要进度指示器，不要放在折叠位置以下 |
| 动画状态转换 | 平滑的 CSS 过渡让智能体感觉更响应 |
| 只高亮一个进行中项目 | 多个 `in_progress` 会使 UI 杂乱 |
| 折叠或暗淡已完成项 | 降低已完成项目的视觉权重 |
| 显示进度百分比 | 一个数字如"67% 完成"一目了然 |
| 保持自动同步 | `stream.values` 响应式更新，无需手动轮询或刷新 |

## 10. 使用场景

| 场景 | 说明 |
|------|------|
| 项目规划 | 智能体将项目分解为任务并依次执行 |
| 研究工作流 | 每个研究问题作为一个 todo |
| 数据处理 | 摄入、验证、转换、导出各一个 todo |
| 上线引导流程 | 智能体逐一完成设置步骤 |
| 报告生成 | 收集数据、分析趋势、撰写摘要、格式化输出 |

---

**参考链接**
- [LangChain Docs — Todo List Pattern](https://docs.langchain.com/deep-agents/frontend/todo-list)
- [useStream API 参考](https://docs.langchain.com/react/use-stream)
- [Deep Agents GitHub](https://github.com/langchain-ai/deep-agents)